# Export all DE comparisons (both projects) → Excel workbook + CSVs

Collects every pydeseq2 DE table from **A04 (organoid/butyrate)** and **A06 (HT29 co-infection)**,
standardises columns to `gene_id, gene_symbol, baseMean, log2FoldChange, lfcSE, stat, pvalue, padj`,
and writes:
- one **Excel workbook** with one sheet per comparison (both projects), plus an INDEX sheet;
- a **folder of CSVs** (one per comparison) and a **zip** of that folder.

Auto-discovers the tables on disk so nothing is missed. Run on the HPC (has openpyxl). No cloud sync needed.

In [1]:
import pandas as pd, numpy as np, os, re, glob, zipfile
from pathlib import Path

A04 = Path(os.environ.get('A04_PROJ', str(Path.home()/'Projects_shared/DECIDE_A04_CdiffOrganoid_Butyrate')))
A06 = Path(os.environ.get('A06_PROJ', str(Path.home()/'Projects_shared/epithelial_coinfection-model_WORKING')))
OUT = A04/'results/DE_export_all'; OUT.mkdir(parents=True, exist_ok=True)
CSVDIR = OUT/'csv'; CSVDIR.mkdir(exist_ok=True)
print('A04 tables:', (A04/'results/tables').exists())
print('A06 DE    :', (A06/'analysis/DE').exists())

A04 tables: True
A06 DE    : True


In [2]:
# ---- discover DE tables in both projects ----
# A04 organoid: results/tables/DE_*.csv  (pydeseq2)
a04_files = sorted(glob.glob(str(A04/'results/tables/DE_*.csv')))
# A06 HT29: analysis/DE/pydeseq2_*_vs_*.csv  (prefer the _annotated versions -> have gene symbols)
a06_all = sorted(glob.glob(str(A06/'analysis/DE/pydeseq2_*_vs_*.csv')))
a06_files=[]; seen=set()
for f in a06_all:
    key=re.sub(r'_annotated','',os.path.basename(f))
    if key in seen: continue
    seen.add(key)
    ann=f.replace('.csv','_annotated.csv') if '_annotated' not in f else f
    a06_files.append(ann if Path(ann).exists() else f)

print('A04 comparisons found:', len(a04_files)); [print('  ',os.path.basename(f)) for f in a04_files]
print('A06 comparisons found:', len(a06_files)); [print('  ',os.path.basename(f)) for f in a06_files]

A04 comparisons found: 7
   DE_Butyrate_10v0_Cd.csv
   DE_Butyrate_10v0_Mock.csv
   DE_Butyrate_5v0_Cd.csv
   DE_Butyrate_5v0_Mock.csv
   DE_Cd_vs_Mock_0mM.csv
   DE_Cd_vs_Mock_10mM.csv
   DE_Cd_vs_Mock_5mM.csv
A06 comparisons found: 6
   pydeseq2_Bt_vs_Cd_annotated.csv
   pydeseq2_Bt_vs_Co_annotated.csv
   pydeseq2_Bt_vs_Mock_annotated.csv
   pydeseq2_Cd_vs_Co_annotated.csv
   pydeseq2_Cd_vs_Mock_annotated.csv
   pydeseq2_Co_vs_Mock_annotated.csv


[None, None, None, None, None, None]

In [3]:
# ---- standardise one DE table to the common schema ----
COMMON=['gene_id','gene_symbol','baseMean','log2FoldChange','lfcSE','stat','pvalue','padj']
def standardise(path, project, comparison):
    d=pd.read_csv(path)
    # identify id / symbol columns
    cols={c2.lower():c2 for c2 in d.columns}
    # gene_id: the index-like column or an ensembl column
    idcol=None
    for cand in ['gene_id','geneid','ensembl','ensembl_gene_id','gene','unnamed: 0','index']:
        if cand in cols: idcol=cols[cand]; break
    if idcol is None:
        d=d.reset_index().rename(columns={'index':'gene_id'}); idcol='gene_id'
    symcol=None
    for cand in ['gene_symbol','symbol','gene_name','genesymbol','hgnc_symbol']:
        if cand in cols: symcol=cols[cand]; break
    out=pd.DataFrame()
    out['gene_id']=d[idcol].astype(str).str.replace(r'\..*','',regex=True)
    out['gene_symbol']=d[symcol].astype(str) if symcol else out['gene_id']
    for k in ['baseMean','log2FoldChange','lfcSE','stat','pvalue','padj']:
        src=None
        for cand in [k, k.lower(), {'log2FoldChange':'log2fc','pvalue':'pval','padj':'adj.pvalue'}.get(k,'')]:
            if cand and cand in cols: src=cols[cand]; break
        out[k]=pd.to_numeric(d[src],errors='coerce') if src else np.nan
    out.insert(0,'project',project); out.insert(1,'comparison',comparison)
    return out

def nice_name(fname, project):
    b=os.path.basename(fname)
    b=re.sub(r'^(DE_|pydeseq2_)','',b); b=re.sub(r'(_annotated)?\.csv$','',b)
    return f'{project}:{b}'

In [4]:
# ---- collate + write ----
tables={}   # sheet/csv name -> df
index_rows=[]
for f in a04_files:
    name=nice_name(f,'A04'); df=standardise(f,'A04',name)
    tables[name]=df; index_rows.append(dict(sheet=name, project='A04 organoid/butyrate', file=os.path.basename(f), n_genes=len(df), n_sig=(df.padj<0.05).sum()))
for f in a06_files:
    name=nice_name(f,'A06'); df=standardise(f,'A06',name)
    tables[name]=df; index_rows.append(dict(sheet=name, project='A06 HT29 co-infection', file=os.path.basename(f), n_genes=len(df), n_sig=(df.padj<0.05).sum()))
index=pd.DataFrame(index_rows)
print(index.to_string(index=False))

# CSVs (one per comparison)
def safe(s): return re.sub(r'[^A-Za-z0-9]+','_',s).strip('_')
for name,df in tables.items():
    df.to_csv(CSVDIR/f'{safe(name)}.csv', index=False)
# zip the CSV folder
zpath=OUT/'DE_all_comparisons_csv.zip'
with zipfile.ZipFile(zpath,'w',zipfile.ZIP_DEFLATED) as z:
    for p in CSVDIR.glob('*.csv'): z.write(p, p.name)
print('\nwrote', len(tables),'CSVs ->', CSVDIR)
print('wrote zip ->', zpath)

                 sheet               project                              file  n_genes  n_sig
  A04:Butyrate_10v0_Cd A04 organoid/butyrate           DE_Butyrate_10v0_Cd.csv    18804   4123
A04:Butyrate_10v0_Mock A04 organoid/butyrate         DE_Butyrate_10v0_Mock.csv    18804   4948
   A04:Butyrate_5v0_Cd A04 organoid/butyrate            DE_Butyrate_5v0_Cd.csv    18804   1328
 A04:Butyrate_5v0_Mock A04 organoid/butyrate          DE_Butyrate_5v0_Mock.csv    18804    264
    A04:Cd_vs_Mock_0mM A04 organoid/butyrate             DE_Cd_vs_Mock_0mM.csv    18804   7605
   A04:Cd_vs_Mock_10mM A04 organoid/butyrate            DE_Cd_vs_Mock_10mM.csv    18804   7540
    A04:Cd_vs_Mock_5mM A04 organoid/butyrate             DE_Cd_vs_Mock_5mM.csv    18804   7918
          A06:Bt_vs_Cd A06 HT29 co-infection   pydeseq2_Bt_vs_Cd_annotated.csv    23605      0
          A06:Bt_vs_Co A06 HT29 co-infection   pydeseq2_Bt_vs_Co_annotated.csv    23605      0
        A06:Bt_vs_Mock A06 HT29 co-infection pydes

In [5]:
# ---- Excel workbook: INDEX + one sheet per comparison ----
xlsx=OUT/'DE_all_comparisons.xlsx'
with pd.ExcelWriter(xlsx, engine='openpyxl') as xw:
    index.to_excel(xw, sheet_name='INDEX', index=False)
    for name,df in tables.items():
        sheet=safe(name)[:31]           # Excel sheet-name limit
        df.to_excel(xw, sheet_name=sheet, index=False)
print('wrote workbook ->', xlsx)
print('sheets:', ['INDEX']+[safe(n)[:31] for n in tables])

wrote workbook -> /home/job37yv/Projects_shared/DECIDE_A04_CdiffOrganoid_Butyrate/results/DE_export_all/DE_all_comparisons.xlsx
sheets: ['INDEX', 'A04_Butyrate_10v0_Cd', 'A04_Butyrate_10v0_Mock', 'A04_Butyrate_5v0_Cd', 'A04_Butyrate_5v0_Mock', 'A04_Cd_vs_Mock_0mM', 'A04_Cd_vs_Mock_10mM', 'A04_Cd_vs_Mock_5mM', 'A06_Bt_vs_Cd', 'A06_Bt_vs_Co', 'A06_Bt_vs_Mock', 'A06_Cd_vs_Co', 'A06_Cd_vs_Mock', 'A06_Co_vs_Mock']


## Notes for colleagues
- Every sheet/CSV has the same columns: `project, comparison, gene_id, gene_symbol, baseMean,
  log2FoldChange, lfcSE, stat, pvalue, padj` (blank where a source table lacked a field).
- **A04** = organoid / *C. difficile* / butyrate (pydeseq2). **A06** = HT29 co-infection (pydeseq2).
- These are the DESeq2/pydeseq2 outputs from each project's pipeline — the same tables the figures draw from.
- Sheet/file names encode project + contrast, e.g. `A04:Cd_vs_Mock_0mM`, `A06:Cd_vs_Co`.
- INDEX sheet lists every comparison with gene count and #significant (padj<0.05) for orientation.